# CNN、MNIST 与错误样本分析

## 学习目标

能够追踪卷积特征图形状，训练真实 MNIST，并分析预测错误。


## 概念模型与执行路径

卷积通过局部连接和权重共享提取空间模式，池化缩小空间尺寸，分类头把特征映射为 logits。交叉熵直接接收 logits。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
import torch
from common.models import ImageClassifier
model = ImageClassifier(channels=1)
x = torch.randn(8, 1, 28, 28)
features = model.features(x)
logits = model(x)
print("input:", x.shape, "features:", features.shape, "logits:", logits.shape)


### 实验 3


In [ ]:
for name, layer in model.features.named_children():
    x = layer(x)
    print(name, layer.__class__.__name__, tuple(x.shape))


### 实验 4


In [ ]:
# 首次运行会下载 MNIST；快速模式只使用小子集。
# python 07-deep-learning/pytorch/examples/train_image_classifier.py --dataset mnist --quick --epochs 3
print("完整训练入口：", PYTORCH_ROOT / "examples/train_image_classifier.py")


### 实验 5


In [ ]:
# 训练后可加载最佳检查点并收集错误样本：
from common.checkpoint import load_checkpoint
checkpoint = PYTORCH_ROOT / "artifacts/image_classifier.pt"
print("checkpoint exists:", checkpoint.exists())


## 底层机制

卷积权重形状为 `(out_channels, in_channels, kernel_h, kernel_w)`。AdaptiveAvgPool 固定分类头输入尺寸，使模型能接受不同但合理的图像尺寸。


## 检查点

逐层解释为何 28x28 图像经过两次 2 倍池化后变为 7x7；再说明本模型为何最终固定为 4x4。


## 试一试

训练一个 quick 模型，画出至少 8 个错误样本及预测类别，按容易混淆的数字对进行归类。


## 常见错误与调试

输入缺少 channel 维、交叉熵前手动 softmax、测试最佳模型前未重新加载检查点、把数据增强应用到验证集。
